# Embeddings

Cada modelo tiene todos los vídeos y la etiqueta en la columna **shoot_zone**,  donde lanzamiento a la derecha es 0,al centro es 1 y a la izquierda es 2. Los vídeos contienen frames hasta el momento del lanzamiento del juagdor al balón, no la secuencia completa de si fue gol o no.

In [ ]:
import os
import pandas as pd

# 1. Directorio con los CSV
data_dir = "Gait_Embeddings_good/"

# 2. Listar sólo los archivos .csv
csv_files = [f for f in os.listdir(data_dir) if f.endswith('.csv')]
print("Archivos encontrados:", csv_files)

# 3. Leer cada CSV en un DataFrame de pandas
dfs = {}
for fname in csv_files:
    path = os.path.join(data_dir, fname)
    dfs[fname] = pd.read_csv(path)

# 4. Explorar cada DataFrame
for name, df in dfs.items():
    print(f"\n=== {name} ===")
    print("Shape:", df.shape)                  # filas × columnas
    print("Columnas:", df.columns.tolist())    # lista de nombres
    print("Primeras 5 filas:")
    print(df.head().to_string(index=False))    # muestra las primeras filas

    # Opcional: ver tipo de datos y memoria
    print("\nInfo:")
    print(df.info())
    print("\nDescripción estadística de columnas numéricas:")
    print(df.describe().T)  # transpuesta para leer mejor



### Carga e inspección inicial de los embeddings (CSV)

En esta celda se cargan **todos los ficheros CSV** del directorio de embeddings y se realiza una inspección rápida para verificar que el formato es el esperado antes de cualquier transformación o experimento. Para cada embedding se muestra:

- Tamaño del dataset (filas × columnas).
- Lista de columnas disponibles (por ejemplo: `video_ID`, `step`, `feat_*`, metadatos y etiqueta).
- Vista previa de las primeras filas.
- Resumen de tipos de datos (`info`) y estadísticas básicas de las columnas numéricas (`describe`).

Este chequeo sirve para detectar de forma temprana problemas típicos (columnas faltantes, tipos incorrectos, valores anómalos o diferencias estructurales entre embeddings) y confirmar que todos los ficheros son coherentes para las fases posteriores.


In [ ]:
from __future__ import annotations

import re
import json
from pathlib import Path
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd


# =========================
# CONFIG
# =========================
EMB_DIR = Path("Gait_Embeddings_good")   # <-- CAMBIA ESTO
PATTERN = "*.csv"                       # si tienes otros nombres, ajusta
REPORT_DIR = Path("embedding_audit_report")
REPORT_DIR.mkdir(exist_ok=True)

# Columnas que suelen estar en tus CSV (ajusta si tienes más)
REQUIRED_BASE_COLS = ["video_ID", "step"]
OPTIONAL_META_COLS = ["pitch_side", "kicker_foot", "shoot_zone"]  # se validan si existen

# Si step debe ser 0..T-1 en todos los videos:
EXPECT_STEP_START_AT_ZERO = True


# =========================
# HELPERS
# =========================
_feat_re = re.compile(r"^feat_(\d+)$")


def find_feat_cols(cols: list[str]) -> list[str]:
    feat = []
    for c in cols:
        m = _feat_re.match(c)
        if m:
            feat.append((int(m.group(1)), c))
    feat.sort(key=lambda x: x[0])
    return [c for _, c in feat]


def check_feat_index_contiguity(feat_cols: list[str]) -> dict:
    """
    Verifica que feat_0..feat_{D-1} existan sin saltos.
    """
    idxs = [int(_feat_re.match(c).group(1)) for c in feat_cols]
    if not idxs:
        return {"ok": False, "reason": "No feat_* columns found", "missing": [], "extra": []}

    D = len(idxs)
    expected = set(range(min(idxs), min(idxs) + D))
    got = set(idxs)
    missing = sorted(expected - got)
    extra = sorted(got - expected)
    ok = (len(missing) == 0 and len(extra) == 0 and min(idxs) == 0)
    reason = None
    if min(idxs) != 0:
        reason = f"feat_* starts at {min(idxs)} (expected feat_0)"
    elif missing:
        reason = f"Missing feat indices: {missing[:10]}{'...' if len(missing) > 10 else ''}"
    elif extra:
        reason = f"Extra/unexpected feat indices: {extra[:10]}{'...' if len(extra) > 10 else ''}"
    return {"ok": ok, "reason": reason, "missing": missing, "extra": extra, "D": D}


def is_integer_series(s: pd.Series) -> bool:
    # Acepta int dtype o floats que sean enteros (p.ej. 3.0)
    if pd.api.types.is_integer_dtype(s):
        return True
    if pd.api.types.is_float_dtype(s):
        vals = s.dropna().to_numpy()
        return np.all(np.isclose(vals, np.round(vals)))
    return False


@dataclass
class FileAuditSummary:
    file: str
    n_rows: int
    n_videos: int
    has_required_cols: bool
    missing_required_cols: list[str]
    D_features: int | None

    nan_total: int
    nan_by_col_top: list[tuple[str, int]]

    step_dtype: str
    step_is_integer_like: bool
    step_has_negatives: bool

    # Por video
    steps_T_unique_count: int
    steps_T_mode: int | None
    videos_with_noncontiguous_steps: int
    videos_with_missing_steps_examples: list[dict]

    duplicated_rows: int

    # Metadatos/labels por video
    meta_inconsistencies: dict

    # Notas / warnings
    warnings: list[str]


def audit_one_csv(path: Path) -> tuple[FileAuditSummary, dict]:
    df = pd.read_csv(path)

    warnings = []

    # Required columns
    cols = df.columns.tolist()
    missing_required = [c for c in REQUIRED_BASE_COLS if c not in cols]
    has_required = (len(missing_required) == 0)

    # Feature columns
    feat_cols = find_feat_cols(cols)
    feat_info = check_feat_index_contiguity(feat_cols)
    D = feat_info.get("D", None)
    if not feat_cols:
        warnings.append("No feat_* columns detected (check column names).")
    elif not feat_info["ok"]:
        warnings.append(f"Feature indexing issue: {feat_info['reason']}")

    # Basic NaN / None
    nan_by_col = df.isna().sum().sort_values(ascending=False)
    nan_total = int(nan_by_col.sum())
    nan_by_col_top = [(idx, int(val)) for idx, val in nan_by_col.head(10).items() if val > 0]

    # Duplicated rows
    duplicated_rows = int(df.duplicated().sum())
    if duplicated_rows > 0:
        warnings.append(f"Found duplicated rows: {duplicated_rows}")

    # Step checks
    if "step" in df.columns:
        step_dtype = str(df["step"].dtype)
        step_is_integer_like = is_integer_series(df["step"])
        step_has_neg = bool((df["step"].dropna() < 0).any())
        if not step_is_integer_like:
            warnings.append("step contains non-integer-like values (decimals that are not .0, or non-numeric).")
        if step_has_neg:
            warnings.append("step contains negative values.")
    else:
        step_dtype = "N/A"
        step_is_integer_like = False
        step_has_neg = False
        warnings.append("Missing 'step' column: cannot validate steps.")

    # Per-video step coverage / contiguity
    videos_with_noncontiguous = 0
    missing_steps_examples = []
    steps_T_mode = None
    steps_T_unique_count = 0

    if has_required:
        # Ensure step as int for checks (only if integer-like)
        tmp = df[["video_ID", "step"]].copy()
        if "step" in tmp.columns and step_is_integer_like:
            tmp["step_int"] = np.round(tmp["step"]).astype(int)
        else:
            tmp["step_int"] = tmp["step"]

        # Group by video
        g = tmp.groupby("video_ID")["step_int"]

        T_per_video = g.nunique()
        steps_T_unique_count = int(T_per_video.nunique())
        if len(T_per_video) > 0:
            steps_T_mode = int(T_per_video.mode().iloc[0])

        # Check contiguity: expected set {0..T-1} (or {min..max})
        for vid, s in g:
            steps = pd.Series(s.dropna().unique())
            if steps.empty:
                videos_with_noncontiguous += 1
                if len(missing_steps_examples) < 8:
                    missing_steps_examples.append({"video_ID": vid, "issue": "No steps found"})
                continue

            steps_sorted = np.sort(steps.to_numpy())

            if EXPECT_STEP_START_AT_ZERO:
                expected = np.arange(0, steps_sorted.max() + 1, dtype=int)
            else:
                expected = np.arange(steps_sorted.min(), steps_sorted.max() + 1, dtype=int)

            missing = np.setdiff1d(expected, steps_sorted)
            if len(missing) > 0:
                videos_with_noncontiguous += 1
                if len(missing_steps_examples) < 8:
                    missing_steps_examples.append({
                        "video_ID": vid,
                        "min_step": int(steps_sorted.min()),
                        "max_step": int(steps_sorted.max()),
                        "n_unique_steps": int(len(steps_sorted)),
                        "missing_steps_first": missing[:15].astype(int).tolist()
                    })
    else:
        warnings.append("Missing video_ID/step -> skipping per-video step validation.")

    # Metadata consistency checks (within each file)
    meta_inconsistencies = {}
    if "video_ID" in df.columns:
        for c in OPTIONAL_META_COLS:
            if c in df.columns:
                # should be constant per video_ID
                nun = df.groupby("video_ID")[c].nunique(dropna=False)
                bad = nun[nun > 1]
                meta_inconsistencies[c] = {
                    "videos_inconsistent": int((nun > 1).sum()),
                    "examples": bad.head(8).to_dict()
                }
                if (nun > 1).any():
                    warnings.append(f"Column '{c}' is not constant within some video_IDs.")
            else:
                meta_inconsistencies[c] = {"present": False}

    summary = FileAuditSummary(
        file=path.name,
        n_rows=int(len(df)),
        n_videos=int(df["video_ID"].nunique()) if "video_ID" in df.columns else 0,
        has_required_cols=has_required,
        missing_required_cols=missing_required,
        D_features=D,
        nan_total=nan_total,
        nan_by_col_top=nan_by_col_top,
        step_dtype=step_dtype,
        step_is_integer_like=bool(step_is_integer_like),
        step_has_negatives=bool(step_has_neg),
        steps_T_unique_count=steps_T_unique_count,
        steps_T_mode=steps_T_mode,
        videos_with_noncontiguous_steps=videos_with_noncontiguous,
        videos_with_missing_steps_examples=missing_steps_examples,
        duplicated_rows=duplicated_rows,
        meta_inconsistencies=meta_inconsistencies,
        warnings=warnings
    )

    details = {
        "feat_cols_count": len(feat_cols),
        "feat_cols_first": feat_cols[:5],
        "feat_cols_last": feat_cols[-5:],
        "feat_index_check": feat_info,
    }

    return summary, details


def audit_all():
    paths = sorted(EMB_DIR.glob(PATTERN))
    if not paths:
        raise FileNotFoundError(f"No CSV files found in {EMB_DIR.resolve()} with pattern {PATTERN}")

    summaries: list[FileAuditSummary] = []
    details_by_file = {}

    # For cross-file checks
    video_sets = {}
    dims = {}
    step_modes = {}

    for p in paths:
        s, d = audit_one_csv(p)
        summaries.append(s)
        details_by_file[p.name] = d

        # Save per-file JSON
        with open(REPORT_DIR / f"{p.stem}_audit.json", "w", encoding="utf-8") as f:
            json.dump({"summary": asdict(s), "details": d}, f, indent=2, ensure_ascii=False)

        # For cross-file comparisons
        # - video_ID set
        try:
            df_ids = pd.read_csv(p, usecols=["video_ID"])
            video_sets[p.name] = set(df_ids["video_ID"].dropna().unique().tolist())
        except Exception:
            video_sets[p.name] = None

        dims[p.name] = s.D_features
        step_modes[p.name] = s.steps_T_mode

    # Cross-file: compare video_ID coverage
    valid_sets = {k: v for k, v in video_sets.items() if isinstance(v, set)}
    common = set.intersection(*valid_sets.values()) if valid_sets else set()
    union = set.union(*valid_sets.values()) if valid_sets else set()

    cross = {
        "n_files": len(paths),
        "video_id_common_count": len(common),
        "video_id_union_count": len(union),
        "files_with_missing_videos": {},
        "feature_dims_by_file": dims,
        "step_mode_by_file": step_modes
    }

    for fname, sset in valid_sets.items():
        missing = sorted(list(common - sset))
        extra = sorted(list(sset - common))
        if missing or extra:
            cross["files_with_missing_videos"][fname] = {
                "missing_from_file_vs_common": len(missing),
                "extra_in_file_vs_common": len(extra),
                "missing_examples": missing[:10],
                "extra_examples": extra[:10]
            }

    # Global summary table
    summary_df = pd.DataFrame([asdict(s) for s in summaries])
    summary_csv = REPORT_DIR / "audit_summary.csv"
    summary_df.to_csv(summary_csv, index=False, encoding="utf-8")

    with open(REPORT_DIR / "audit_cross_file.json", "w", encoding="utf-8") as f:
        json.dump(cross, f, indent=2, ensure_ascii=False)

    # Pretty print to console (lo importante)
    cols_show = [
        "file", "n_rows", "n_videos", "D_features",
        "nan_total", "duplicated_rows",
        "step_is_integer_like", "step_has_negatives",
        "steps_T_unique_count", "steps_T_mode",
        "videos_with_noncontiguous_steps"
    ]
    print("\n=== AUDIT SUMMARY (por fichero) ===")
    print(summary_df[cols_show].to_string(index=False))

    print("\n=== CROSS-FILE CHECKS ===")
    print(f"Files: {cross['n_files']}")
    print(f"Common video_IDs across files: {cross['video_id_common_count']}")
    print(f"Union video_IDs across files:  {cross['video_id_union_count']}")
    if cross["files_with_missing_videos"]:
        print("\nFiles with video_ID differences vs common:")
        for k, v in cross["files_with_missing_videos"].items():
            print(f" - {k}: missing={v['missing_from_file_vs_common']}, extra={v['extra_in_file_vs_common']}")
    else:
        print("All files share the same video_ID set (good).")

    print(f"\nSaved report folder: {REPORT_DIR.resolve()}")
    print(f" - {summary_csv.name}")
    print(" - per-file *_audit.json + audit_cross_file.json")


if __name__ == "__main__":
    audit_all()


En esta celda se calcula (i) el número de vídeos únicos y la distribución de clases (shoot_zone) por video_ID para confirmar el desbalance, (ii) el número de dimensiones feat_* en cada CSV para verificar la dimensionalidad de los embeddings, y (iii) el número de steps por vídeo para comprobar que cada extractor produce secuencias con longitud esperada (constante o variable). Este chequeo sirve como validación previa para asegurar consistencia antes de fusionar embeddings o aplicar la selección de steps.

In [ ]:
# Agrupar por video_ID y tomar el primer valor de shoot_zone de cada vídeo
num_videos = df['video_ID'].nunique()
print(f"Número de videos: {num_videos}\n")

video_shoot_zones = df.groupby('video_ID')['shoot_zone'].first()
# Contar cuántos vídeos hay en cada zona de tiro
zone_counts = video_shoot_zones.value_counts().sort_index()

# Mostrar resultados con etiquetas legibles
zone_labels = {0: 'derecha', 1: 'centro', 2: 'izquierda'}
for zone, count in zone_counts.items():
    print(f"{zone_labels.get(zone, zone)}: {count} vídeos")
print("\n")

# Cuantos features hay en cada CSV, cada col con nombre feat_ + número en CSV
feature_counts = {}
for name, df in dfs.items():
    feature_counts[name] = df.shape[1] - 5  # -5 para video_ID, shoot_zone, pitch_side, kicker_foot

print("\nFeature counts:")
for name, count in feature_counts.items():
    print(f"{name}: {count} features")
print("\n")

# Franjas horizontales por vídeo en cada CSV, no son son los frames del vídeo, sino partes horizontales del cuerpo (trazos) extraídas tras analizar todo el vídeo completo.
for name, df in dfs.items():
    steps_per_video = df['video_ID'].value_counts()
    unique_frame_counts = steps_per_video.unique()
    if len(unique_frame_counts) == 1:
        print(f"CSV: {name} => {unique_frame_counts[0]} steps por vídeo.")
    else:
        print(f"CSV: {name} => {steps_per_video.value_counts(1).round(4).to_dict()} steps por vídeo.")
print("\n")

## Detectar steps con mayor información

### Análisis de relevancia de steps y variables obtenidas

Para estimar qué franjas corporales (*steps*) aportan mayor información discriminante en los embeddings, se aplicaron dos enfoques complementarios: uno **estadístico** y otro **empírico**. Cada enfoque genera variables específicas que permiten cuantificar la relevancia relativa de cada step dentro de la secuencia.

Los resultados se almacenan en un único archivo CSV, donde cada fila representa un step corporal y cada columna las variables derivadas del análisis. Además, se generan visualizaciones en forma de barras y líneas superpuestas: las barras muestran la varianza normalizada y los rendimientos medios de cada modelo, mientras que la línea indica el valor del *hybrid score*, facilitando la comparación visual entre enfoques y la identificación de las zonas más informativas del cuerpo.  




### Análisis empírico por steps mediante Logistic Regression y Random Forest 

El análisis empírico se diseñó para evaluar, de manera individual, la capacidad predictiva de cada región corporal en los embeddings mediante dos clasificadores complementarios: **Regresión Logística (Logistic Regression)** y **Random Forest**.  

La **Regresión Logística** se emplea como modelo lineal de referencia, proporcionando una medida directa de la **separabilidad lineal** entre clases en el espacio de características asociado a cada step. Esto permite observar en qué regiones del cuerpo las representaciones embebidas permiten una discriminación clara sin necesidad de transformaciones no lineales.  

Por su parte, el **Random Forest** introduce un enfoque **no lineal**, capaz de modelar interacciones complejas entre características dentro del mismo step. Su inclusión permite capturar relaciones estructurales más ricas entre las variables y contrastar si los patrones detectados por el modelo lineal se mantienen cuando se consideran dependencias de orden superior.  

Ambos modelos se entrenan y evalúan de forma independiente para cada step y para distintas semillas aleatorias, calculando los valores de F1 normalizados y sus desviaciones estándar correspondientes. Este procedimiento se repite sistemáticamente para todos los steps y los resultados se integran posteriormente en un análisis de consenso, lo que permite obtener una medida estable y reproducible de la relevancia empírica de cada franja corporal.  

Finalmente, el rendimiento medio de ambos modelos se fusiona con el análisis estadístico mediante el cálculo del **`hybrid_global_score`**, generando un indicador combinado que resume simultáneamente la informatividad intrínseca de los embeddings y su efectividad práctica en la predicción.


| Variable                | Tipo                         | Cómo se calcula / Fórmula                                                             | Rango típico                       | Interpretación práctica                                                                       |
| ----------------------- | ---------------------------- | ------------------------------------------------------------------------------------- | ---------------------------------- | --------------------------------------------------------------------------------------------- |
| `step`                  | Índice                       | Identificador de la franja corporal (0=cabeza → N=pies).                              | entero                             | Eje vertical (o horizontal) en los heatmaps.                                                  |
| `mean_variance`         | Estadístico (sin normalizar) | Media de la varianza de todas las columnas `feat_*` en ese `step`.                    | ≥0 (depende de escala de features) | Dispersión intrínseca de embeddings en el step. Mayor → más diversidad/información potencial. |
| `statistical_norm`      | Estadístico normalizado      | Min–max sobre `mean_variance` entre steps del **mismo CSV**:  ((x-\min)/(\max-\min)). | [0, 1]                             | Importancia estadística relativa del step dentro del embedding/semilla. 1 = el más variable.  |
| `f1_logistic`           | Empírico bruto               | F1 macro de **Logistic Regression** entrenado solo con ese `step`.                    | [0, 1]                             | Capacidad de separación **lineal** por step.                                                  |
| `f1_logistic_norm`      | Empírico normalizado         | Min–max de `f1_logistic` entre steps de ese CSV.                                      | [0, 1]                             | Importancia empírica **lineal** relativa del step.                                            |
| `f1_rf`                 | Empírico bruto               | F1 macro de **Random Forest** con ese `step`.                                         | [0, 1]                             | Capacidad de separación **no lineal** por step.                                               |
| `f1_rf_norm`            | Empírico normalizado         | Min–max de `f1_rf` entre steps de ese CSV.                                            | [0, 1]                             | Importancia empírica **no lineal** relativa del step.                                         |
| `hybrid_logistic_score` | Híbrido (por semilla)        | (\alpha \cdot \text{statistical_norm} + (1-\alpha)\cdot \text{f1_logistic_norm})      | [0, 1]                             | Score que combina variabilidad + rendimiento lineal.                                          |
| `hybrid_rf_score`       | Híbrido (por semilla)        | (\alpha \cdot \text{statistical_norm} + (1-\alpha)\cdot \text{f1_rf_norm})            | [0, 1]                             | Score que combina variabilidad + rendimiento no lineal.                                       |
| `seed`                  | Meta                         | Semilla usada en `train_test_split` (y RF).                                           | —                                  | Para trazar estabilidad entre particiones aleatorias.                                         |


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split


def draw_heatmaps(df: pd.DataFrame, out_png: str):
    """Dibuja y guarda los tres heatmaps con tamaño adaptado al nº de steps."""
    df_vis = df.sort_values("step", ascending=True).reset_index(drop=True)
    n_steps = len(df_vis)
    base_width = 15
    base_height = max(6, n_steps / 3)  # escala automática según steps

    plt.figure(figsize=(base_width, base_height))

    # Logistic
    plt.subplot(1, 3, 1)
    sns.heatmap(df_vis[["f1_logistic_norm"]], cmap="YlOrRd",
                cbar=True, annot=True, fmt=".2f", linewidths=.5)
    plt.title("Empírico Normalizado (Logistic Regression)")
    plt.ylabel("Step (0=cabeza, N=pies)")

    # RF
    plt.subplot(1, 3, 2)
    sns.heatmap(df_vis[["f1_rf_norm"]], cmap="BuGn",
                cbar=True, annot=True, fmt=".2f", linewidths=.5)
    plt.title("Empírico Normalizado (Random Forest)")
    plt.ylabel("")

    # Varianza
    plt.subplot(1, 3, 3)
    sns.heatmap(df_vis[["statistical_norm"]], cmap="Blues",
                cbar=True, annot=True, fmt=".2f", linewidths=.5)
    plt.title("Estadístico Normalizado (Varianza media)")
    plt.ylabel("")

    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.close()
    print(f" Heatmap guardado: {out_png}")


def step_analysis_base(input_csv: str, output_dir: str, seed: int = 42, n_estimators: int = 2000, heatmaps: bool = False):
    """
    Calcula la varianza media por step y los F1-scores empíricos (LogReg y RandomForest).
    Lógica:
      - Si existe CSV y NO existe heatmap -> generar heatmap desde el CSV.
      - Si existen CSV y heatmap -> omitir recálculo.
      - Si no existe CSV -> calcular, guardar CSV y heatmap.
    """
    os.makedirs(output_dir, exist_ok=True)
    output_csv = os.path.join(output_dir, f"step_analysis_seed{seed}.csv")
    heatmap_path = os.path.join(output_dir, f"heatmap_seed{seed}.png")

    # Caso 1: existe CSV y NO existe heatmap -> pintar desde CSV
    if os.path.exists(output_csv) and not os.path.exists(heatmap_path):
        df_loaded = pd.read_csv(output_csv)
        if heatmaps == True:
            draw_heatmaps(df_loaded, heatmap_path)
            print(f"🖼️ CSV encontrado pero faltaba heatmap (seed={seed}). Se genera desde CSV.")
        else:
            print(" Omitido dibujo de heatmaps (heatmaps=False).")
        return df_loaded

    # Caso 2: existen CSV y heatmap -> omitir
    if os.path.exists(output_csv) and os.path.exists(heatmap_path):
        print(f"⚡ Resultados ya existen para {os.path.basename(input_csv)} (seed={seed}) → omitido.")
        return pd.read_csv(output_csv)

    # Caso 3: no existe CSV -> calcular
    print(f"\n🔹 Ejecutando análisis base para: {os.path.basename(input_csv)} (seed={seed})")

    # === Cargar datos ===
    df = pd.read_csv(input_csv)
    feature_cols = [c for c in df.columns if c.startswith("feat_")]
    steps = sorted(df["step"].unique())

    # === 1) Varianza media por step (estadístico) ===
    statistical_importance = (
        df.groupby("step")[feature_cols].var().mean(axis=1)
          .rename("mean_variance").reset_index()
    )
    statistical_importance["statistical_norm"] = (
        (statistical_importance["mean_variance"] - statistical_importance["mean_variance"].min()) /
        (statistical_importance["mean_variance"].max() - statistical_importance["mean_variance"].min())
    )

    # === 2) F1 por step (Logistic Regression) ===
    f1_logistic_per_step = []
    for s in steps:
        df_s = df[df["step"] == s]
        X = df_s[feature_cols].values
        y = df_s["shoot_zone"].values
        if len(np.unique(y)) < 2:
            continue
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.1, stratify=y, random_state=seed
        )
        clf = LogisticRegression(max_iter=1000, n_jobs=-1, class_weight="balanced", random_state=0)
        clf.fit(X_train, y_train)
        f1 = f1_score(y_test, clf.predict(X_test), average="macro")
        f1_logistic_per_step.append((s, f1))

    df_logistic = pd.DataFrame(f1_logistic_per_step, columns=["step", "f1_logistic"])
    df_logistic["f1_logistic_norm"] = (
        (df_logistic["f1_logistic"] - df_logistic["f1_logistic"].min()) /
        (df_logistic["f1_logistic"].max() - df_logistic["f1_logistic"].min())
    )

    # === 3) F1 por step (Random Forest) ===
    f1_rf_per_step = []
    for s in steps:
        df_s = df[df["step"] == s]
        X = df_s[feature_cols].values
        y = df_s["shoot_zone"].values
        if len(np.unique(y)) < 2:
            continue
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.1, stratify=y, random_state=seed
        )
        rf = RandomForestClassifier(n_estimators=n_estimators, random_state=seed, n_jobs=-1, class_weight="balanced")
        rf.fit(X_train, y_train)
        f1 = f1_score(y_test, rf.predict(X_test), average="macro")
        f1_rf_per_step.append((s, f1))

    df_rf = pd.DataFrame(f1_rf_per_step, columns=["step", "f1_rf"])
    df_rf["f1_rf_norm"] = (
        (df_rf["f1_rf"] - df_rf["f1_rf"].min()) /
        (df_rf["f1_rf"].max() - df_rf["f1_rf"].min())
    )

    # === 4) Fusionar y guardar CSV ===
    results_combined = (
        statistical_importance
        .merge(df_logistic, on="step", how="inner")
        .merge(df_rf, on="step", how="inner")
    )
    results_combined.to_csv(output_csv, index=False)
    print(f" Guardado: {output_csv}")

    # === 5) Heatmaps adaptativos ===
    if heatmaps == True:
        draw_heatmaps(results_combined, heatmap_path)
    else:
        print(" Omitido dibujo de heatmaps (heatmaps=False).")
    # === 6) Correlación ===
    corr = results_combined["f1_logistic_norm"].corr(results_combined["f1_rf_norm"])
    print(f"🔹 Correlación Logistic–RF (por step): {corr:.3f}")

    return results_combined


# === Ejemplo de uso ===
if __name__ == "__main__":
    data_dir = "Gait_Embeddings_good/"
    csv_files = [f for f in os.listdir(data_dir) if f.endswith(".csv")]
    seeds = [1, 42, 77, 123, 567, 1000, 1467, 2003, 3001, 4009, 5001, 6007, 7003, 7812, 8008, 9001, 9500, 9956, 11111, 12345]


    for embedding_csv in csv_files:
        out_dir = f"Step_Analysis_Selection/{embedding_csv.replace('.csv', '')}"
        for seed in seeds:
            step_analysis_base(
                input_csv=os.path.join(data_dir, embedding_csv),
                output_dir=out_dir,
                seed=seed,
                n_estimators=2000,
                heatmaps = False
            )


### Análisis consensuado de importancia entre semillas

Con el objetivo de identificar qué franjas corporales aportan mayor información discriminante, se evaluó cada step de forma independiente mediante dos clasificadores complementarios: una Regresión Logística (modelo lineal) y un Random Forest (modelo no lineal). Para cada semilla se calculó el F1 macro por step y se normalizó dentro de cada embedding (min–max entre steps), de forma que las puntuaciones reflejan importancia relativa y no rendimiento absoluto. Posteriormente, se obtuvo un consenso agregando media y desviación típica de dichas puntuaciones a lo largo de múltiples semillas, lo que permite cuantificar tanto la relevancia promedio de cada región como su sensibilidad a la partición de datos.

Finalmente, se definió un índice híbrido por step (hybrid_global_score) combinando la señal empírica lineal y no lineal con un término estadístico basado en la varianza normalizada (α, β, γ). Este score se utilizó para seleccionar los top-K steps de cada embedding y construir versiones recortadas, reduciendo la longitud efectiva de la secuencia sin perder, a priori, las regiones más informativas. En conjunto, este procedimiento proporciona un criterio sistemático y reproducible para la selección de franjas corporales antes de los experimentos de V4.

| Variable               | Qué es realmente (según tu código)                                                                                                         | Rango  | Interpretación correcta                                                                                              |
| ---------------------- | ------------------------------------------------------------------------------------------------------------------------------------------ | ------ | -------------------------------------------------------------------------------------------------------------------- |
| `step`                 | Índice de franja corporal                                                                                                                  | entero | 0≈parte alta → N≈parte baja (según tu convención)                                                                    |
| `f1_logistic_mean`     | Media **entre semillas** de `f1_logistic_norm`                                                                                             | [0,1]  | Importancia empírica **lineal relativa** del step (promedio en múltiples particiones)                                |
| `f1_logistic_std`      | Desviación típica **entre semillas** de `f1_logistic_norm`                                                                                 | ≥0     | **Estabilidad entre semillas** del ranking lineal (bajo = más robusto)                                               |
| `f1_rf_mean`           | Media **entre semillas** de `f1_rf_norm`                                                                                                   | [0,1]  | Importancia empírica **no lineal relativa** del step                                                                 |
| `f1_rf_std`            | Desviación típica **entre semillas** de `f1_rf_norm`                                                                                       | ≥0     | Estabilidad entre semillas del ranking no lineal                                                                     |
| `hybrid_logistic_mean` | Media entre semillas de `hybrid_logistic_score`                                                                                            | [0,1]  | Score híbrido (varianza + LR) promedio entre seeds                                                                   |
| `hybrid_logistic_std`  | Std entre semillas de `hybrid_logistic_score`                                                                                              | ≥0     | Estabilidad entre semillas del híbrido con LR                                                                        |
| `hybrid_rf_mean`       | Media entre semillas de `hybrid_rf_score`                                                                                                  | [0,1]  | Score híbrido (varianza + RF) promedio entre seeds                                                                   |
| `hybrid_rf_std`        | Std entre semillas de `hybrid_rf_score`                                                                                                    | ≥0     | Estabilidad entre semillas del híbrido con RF                                                                        |
| `statistical_mean`     | Media entre semillas de `statistical_norm`                                                                                                 | [0,1]  | En tu caso debería ser prácticamente el mismo valor siempre (porque la varianza no depende del split)                |
| `statistical_std`      | Std entre semillas de `statistical_norm`                                                                                                   | ~0     | Debería ser ~0 (si no, es por ruido numérico)                                                                        |
| `hybrid_global_score`  |  En el código es: `alpha*statistical_mean + beta*f1_logistic_mean + gamma*f1_rf_mean` | [0,1]  | Score final por step que combina: variabilidad + capacidad lineal + capacidad no lineal (ya promediadas entre seeds) |


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os


def step_analysis_hybrid(
    input_csv: str,
    output_dir: str,
    alpha: float = 0.3,
    beta: float = 0.35,
    gamma: float = 0.35,
    seeds: list = [42],
    force_recompute: bool = False
):
    """
    🔹 Versión híbrida completa.
    Mantiene los híbridos individuales (varianza + modelo)
    pero calcula el score global mediante la fusión directa (α·Var + β·Log + γ·RF).
    """

    os.makedirs(output_dir, exist_ok=True)
    all_results = []

    # ===  Obtener resultados base por semilla ===
    for seed in seeds:
        base_csv = os.path.join(output_dir, f"step_analysis_seed{seed}.csv")

        if not os.path.exists(base_csv) or force_recompute:
            print(f"⚙️ Ejecutando análisis base (seed={seed})...")
            df_seed = step_analysis_base(input_csv, output_dir, seed=seed)
        else:
            print(f" CSV base encontrado para seed={seed}")
            df_seed = pd.read_csv(base_csv)

        # Calcular híbridos individuales
        df_seed["hybrid_logistic_score"] = (
            alpha * df_seed["statistical_norm"] + (1 - alpha) * df_seed["f1_logistic_norm"]
        )
        df_seed["hybrid_rf_score"] = (
            alpha * df_seed["statistical_norm"] + (1 - alpha) * df_seed["f1_rf_norm"]
        )
        df_seed["seed"] = seed
        all_results.append(df_seed)

    # === Combinar semillas ===
    df_all = pd.concat(all_results, ignore_index=True)
    print(f"\n Combinando {len(seeds)} semillas ({len(df_all)} filas totales)")

    # === Calcular consenso medio y std ===
    consensus = (
        df_all.groupby("step")[["f1_logistic_norm", "f1_rf_norm",
                                "hybrid_logistic_score", "hybrid_rf_score",
                                "statistical_norm"]]
        .agg(["mean", "std"])
        .reset_index()
    )

    consensus.columns = [
        "step",
        "f1_logistic_mean", "f1_logistic_std",
        "f1_rf_mean", "f1_rf_std",
        "hybrid_logistic_mean", "hybrid_logistic_std",
        "hybrid_rf_mean", "hybrid_rf_std",
        "statistical_mean", "statistical_std"
    ]

    # ===  Calcular score híbrido global (fusión directa)
    if not np.isclose(alpha + beta + gamma, 1.0):
        raise ValueError("Los pesos α + β + γ deben sumar 1.0")

    consensus["hybrid_global_score"] = (
        alpha * consensus["statistical_mean"]
        + beta  * consensus["f1_logistic_mean"]
        + gamma * consensus["f1_rf_mean"]
    )
    # redondear todos los scores
    consensus["hybrid_global_score"] = consensus["hybrid_global_score"].round(4)
    consensus["statistical_mean"] = consensus["statistical_mean"].round(4)
    consensus["statistical_std"] = consensus["statistical_std"].round(4)
    consensus["f1_logistic_std"] = consensus["f1_logistic_std"].round(4)
    consensus["f1_logistic_mean"] = consensus["f1_logistic_mean"].round(4)
    consensus["f1_rf_mean"] = consensus["f1_rf_mean"].round(4)
    consensus["f1_rf_std"] = consensus["f1_rf_std"].round(4)
    consensus["hybrid_logistic_mean"] = consensus["hybrid_logistic_mean"].round(4)
    consensus["hybrid_rf_mean"] = consensus["hybrid_rf_mean"].round(4)
    consensus["hybrid_logistic_std"] = consensus["hybrid_logistic_std"].round(4)
    consensus["hybrid_rf_std"] = consensus["hybrid_rf_std"].round(4)




    # ===  Guardar CSV final ===
    out_csv = os.path.join(output_dir, "step_selection_hybrid_final.csv")
    consensus.to_csv(out_csv, index=False)
    print(f" CSV híbrido global guardado en: {out_csv}")

    # ===  Heatmaps adaptativos (SOLO 3) ===
    df_vis = consensus.sort_values("step", ascending=False).reset_index(drop=True)
    n_steps = len(df_vis)

    base_width = 16  # un poco menos ancho al haber 3 plots
    base_height = max(10, n_steps / 3)

    plt.figure(figsize=(base_width, base_height))

    # 1) Varianza
    plt.subplot(1, 3, 1)
    sns.heatmap(
        df_vis[["statistical_mean"]],
        cmap="Blues", annot=True, fmt=".2f", linewidths=0.5,
        cbar_kws={"fraction": 0.04}, annot_kws={"size": 14}
    )
    ax = plt.gca()
    ax.tick_params(axis="x", labelsize=16)
    plt.title("Varianza normalizada", fontsize=20)
    plt.ylabel("Step (0=cabeza \u2192 N=pies)", fontsize=16)
    

    # 2) F1 medios (LR vs RF)
    plt.subplot(1, 3, 2)
    sns.heatmap(
        df_vis[["f1_logistic_mean", "f1_rf_mean"]],
        cmap="YlOrRd", annot=True, fmt=".2f", linewidths=0.5,
        cbar_kws={"fraction": 0.04}, annot_kws={"size": 14}
    )
    ax = plt.gca()
    ax.tick_params(axis="x", labelsize=16)

    plt.title("F1 medio (LR vs RF)", fontsize=16)
    plt.ylabel("")

    # 3) Score global
    plt.subplot(1, 3, 3)
    sns.heatmap(
        df_vis[["hybrid_global_score"]],
        cmap="Greens", annot=True, fmt=".2f", linewidths=0.5,
        cbar_kws={"fraction": 0.04}, annot_kws={"size": 14},
    )
    ax = plt.gca()
    ax.tick_params(axis="x", labelsize=16)
    
    plt.title(f"Score global (\u03B1={alpha}, \u03B2={beta}, \u03B3={gamma})", fontsize=18)
    plt.ylabel("")

    plt.tight_layout()
    out_fig = os.path.join(output_dir, "step_selection_heatmaps.png")
    plt.savefig(out_fig, dpi=300)
    plt.close()
    print(f" Heatmaps guardados en: {out_fig}")



data_dir = "Gait_Embeddings_good/"
csv_files = [f for f in os.listdir(data_dir)]
seeds = [1, 42, 77, 123, 567, 1000, 1467, 2003, 3001, 4009, 5001, 6007, 7003, 7812, 8008, 9001, 9500, 9956, 11111, 12345]
for embedding_csv in csv_files:
    step_analysis_hybrid(
        input_csv=os.path.join(data_dir, embedding_csv),
        output_dir=f"Step_Analysis_Selection/" + embedding_csv.replace('.csv', ''),
        alpha=0.2, beta=0.4, gamma=0.4,
        seeds=seeds
    )


### Interpretación de selección de steps

Para evaluar la estabilidad de los resultados, se emplearon métricas estadísticas robustas —la mediana, el rango intercuartílico (IQR) y el coeficiente de variación (CV)—, que ofrecen una estimación más fiable al ser menos sensibles a valores atípicos.
Estas medidas permiten cuantificar la variabilidad relativa del rendimiento entre distintas semillas o pasos (steps), proporcionando una visión más estable del comportamiento de cada modelo.

La clasificación de los resultados en estables, intermedios o inestables se realiza de forma adaptativa y basada en datos, utilizando los percentiles 25 y 75 (P25–P75) de la distribución global de la variabilidad.
Este enfoque evita el uso de umbrales fijos y garantiza una evaluación más justa y coherente con la magnitud real del ruido presente en los datos.

In [ ]:
import os
import pandas as pd
import numpy as np

ROOT = "Step_Analysis_Selection"  # carpeta con subcarpetas por embedding
TARGET_CSV = "step_selection_hybrid_final.csv"

def robust_stats(series):
    series = series.dropna()
    med = series.median()
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    sd = series.std(ddof=1)
    mean = series.mean()
    cv = sd/mean if mean > 0 else np.nan
    return pd.Series({"median": med, "iqr": iqr, "std": sd, "mean": mean, "cv": cv})

rows = []
for emb in sorted(os.listdir(ROOT)):
    csv_path = os.path.join(ROOT, emb, TARGET_CSV)
    if not os.path.isfile(csv_path):
        continue
    df = pd.read_csv(csv_path)

    # Métricas a evaluar (puedes añadir 'hybrid_logistic_mean', 'hybrid_rf_mean', etc.)
    metrics = {
        "F1 Logistic": "f1_logistic_mean",
        "F1 RF": "f1_rf_mean",
        "Hybrid Global": "hybrid_global_score",
        "Statistical": "statistical_mean" if "statistical_mean" in df.columns else "statistical_norm",
    }

    for nice, col in metrics.items():
        s = df[col]
        stats = robust_stats(s)
        rows.append({
            "embedding": emb,
            "metric": nice,
            "median": stats["median"],
            "mean": stats["mean"],
            "std": stats["std"],
            "iqr": stats["iqr"],
            "cv": stats["cv"]
        })

summary = pd.DataFrame(rows)

# Cortes por percentiles globales (data-driven)
def label_by_percentiles(df, col, low=0.25, high=0.75):
    lo = df[col].quantile(low)
    hi = df[col].quantile(high)
    def lab(v):
        if pd.isna(v): return "—"
        if v <= lo: return " Estable"
        if v >= hi: return "❌ Inestable"
        return " Intermedia"
    return lo, hi, df[col].apply(lab)

# Usa CV como métrica base de estabilidad (puedes combinar con IQR si quieres)
lo_cv, hi_cv, labels = label_by_percentiles(summary, "cv", 0.25, 0.75)
summary["stability_cv"] = labels

# (Opcional) combinación con IQR: etiqueta final = peor de las dos
lo_iqr, hi_iqr, labels_iqr = label_by_percentiles(summary, "iqr", 0.25, 0.75)
def combine(a, b):
    order = {" Estable": 0, " Intermedia": 1, "❌ Inestable": 2, "—": 3}
    return a if order[a] >= order[b] else b
summary["stability_final"] = [combine(a, b) for a, b in zip(summary["stability_cv"], labels_iqr)]

print(f"Cortes CV -> P25={lo_cv:.3f}, P75={hi_cv:.3f} | Cortes IQR -> P25={lo_iqr:.3f}, P75={hi_iqr:.3f}")
out_path = os.path.join(ROOT, "stability_summary_robust.csv")
summary.to_csv(out_path, index=False)
print(" Resumen robusto guardado en:", out_path)


## V4 - V5 - Combinación de embeddings y Selección de Steps 

### V4: Fusión por Media (mean) vs. Concatenación (concat)

**Fusión de embeddings del mismo modelo (distintas bases):**
En este método se combinan los embeddings generados por un mismo modelo de extracción (por ejemplo, GaitGL o GaitSet), pero entrenado sobre distintas bases de datos como OUMVLP, GREW o CASIA-B. Estos modelos comparten la misma arquitectura, el mismo número de steps (franjas horizontales del cuerpo) y la misma dimensionalidad de features, por lo que sus embeddings son completamente compatibles. La fusión se realiza concatenando las features correspondientes al mismo step de cada vídeo, de modo que cada franja del cuerpo mantiene su posición original pero incorpora información adicional de las distintas bases. Este proceso enriquece cada representación espacial del movimiento, aumentando la robustez y la capacidad de generalización del modelo de predicción final.

**Fusión por Media (mean) vs. Concatenación (concat):**
La elección entre fusionar embeddings por media (mean) o por concatenación (concat) depende fundamentalmente de la arquitectura del modelo que los va a utilizar. La fusión por media calcula un vector "consenso" promediando las características de varios embeddings para cada step. Este proceso reduce el ruido y crea una representación más robusta y generalista, pero a costa de diluir información específica que pudiera ser única de un embedding. De esta técnica se beneficia principalmente el MLP, un modelo que no procesa secuencias y que rinde mejor con un único vector de entrada de alta calidad y dimensionalidad contenida.

Por otro lado, la concatenación apila los vectores de características uno al lado del otro, creando un "súper-vector" que preserva toda la información original de cada embedding, aunque aumenta drásticamente la dimensionalidad. De este método se benefician los modelos secuenciales más complejos, como la LSTM y la TCN. Estas arquitecturas tienen la capacidad de aprender por sí mismas qué características, de entre los cientos disponibles, son las más relevantes para cada parte de la secuencia, sin perder los patrones sutiles que el promediado podría haber eliminado.

Dentro de una fila, el vector [feat_0 … feat_D-1] es un embedding: cada feat_i es una dimensión latente de ese espacio. Esas dimensiones no están ordenadas temporalmente ni suelen tener un significado interpretable individual; son ejes aprendidos por el modelo.

In [ ]:
import pandas as pd
import numpy as np
import os

# === Modo de combinación por defecto: "concat" o "mean" ===
COMBINE_MODE = "mean"  # "concat" | "mean"

# === Directorios ===
input_dir = "Gait_Embeddings_good"
output_dir = "Gait_Embeddings_Combined/mean2"
os.makedirs(output_dir, exist_ok=True)

# === Rutas de los CSVs ===
csv_paths = {
    "baseline_CASIAB": os.path.join(input_dir, "baseline_CASIAB.csv"),
    "baseline_OUMVLP": os.path.join(input_dir, "baseline_OUMVLP.csv"),
    "gaitgl_GREW": os.path.join(input_dir, "gaitgl_GREW.csv"),
    "gaitgl": os.path.join(input_dir, "gaitgl.csv"),
    "gaitgl_GREW_BNNeck": os.path.join(input_dir, "gaitgl_GREW_BNNeck.csv"),
    "gaitgl_OUMVLP": os.path.join(input_dir, "gaitgl_OUMVLP.csv"),
    "gaitpart_GREW": os.path.join(input_dir, "gaitpart_GREW.csv"),
    "gaitpart_OUMVLP": os.path.join(input_dir, "gaitpart_OUMVLP.csv"),
    "gaitset": os.path.join(input_dir, "gaitset.csv"),
    "gaitset_GREW": os.path.join(input_dir, "gaitset_GREW.csv"),
    "gaitset_OUMVLP": os.path.join(input_dir, "gaitset_OUMVLP.csv"),
    "gln_phase1": os.path.join(input_dir, "gln_phase1.csv"),
    "gln_phase2": os.path.join(input_dir, "gln_phase2.csv")
}

compatible_groups = {
    "baseline_31steps": ["baseline_CASIAB", "baseline_OUMVLP"],
    "gaitpart_16steps": ["gaitpart_GREW", "gaitpart_OUMVLP"],
    "gaitset_62steps": ["gaitset", "gaitset_GREW", "gaitset_OUMVLP"],
    "gaitgl_64steps": ["gaitgl_GREW", "gaitgl_GREW_BNNeck", "gaitgl_OUMVLP"],
    "gln_93steps": ["gln_phase1", "gln_phase2"],
}

ID_COLS = ["step", "video_ID", "pitch_side", "kicker_foot", "shoot_zone"]

def combinar_grupo(nombre_grupo, lista_csv, mode="concat"):
    assert mode in {"concat", "mean"}, "mode debe ser 'concat' o 'mean'"
    print(f"\n🔹 Combinando grupo: {nombre_grupo}  |  modo={mode}")
    dfs_prefixed = []   # para concat
    mats = []           # para mean (matrices de features sin prefijo)
    feats_counts = []   # nº features por fuente (para mean)
    ids_ref = None      # IDs de referencia para validar alineación

    for idx, csv_name in enumerate(lista_csv):
        path = csv_paths[csv_name]
        df = pd.read_csv(path)

        # columnas de features
        feat_cols = [c for c in df.columns if c.startswith("feat_")]
        if not feat_cols:
            raise ValueError(f"No se encontraron columnas 'feat_*' en {csv_name}")

        # verificar columnas ID
        for col in ID_COLS:
            if col not in df.columns:
                raise ValueError(f"Columna {col} faltante en {csv_name}")

        # referencia de IDs y orden
        if ids_ref is None:
            ids_ref = df[ID_COLS].copy()
        else:
            if not ids_ref.equals(df[ID_COLS]):
                raise ValueError(
                    f"El orden/valores de {ID_COLS} no coincide entre {lista_csv[0]} y {csv_name}"
                )

        # preparar datos según modo
        if mode == "concat":
            # mantenemos un prefijo temporal para no colisionar al concatenar
            rename_map = {c: f"{csv_name}__{i}" for i, c in enumerate(feat_cols)}
            df_pref = df.rename(columns=rename_map)
            dfs_prefixed.append(df_pref[ID_COLS + list(rename_map.values())])
        elif mode == "mean":
            mats.append(df[feat_cols].to_numpy(dtype=np.float32))
            feats_counts.append(len(feat_cols))

    if mode == "concat":
        # concatenación horizontal
        df_merged = dfs_prefixed[0]
        for other_df in dfs_prefixed[1:]:
            other_feats = [c for c in other_df.columns if "__" in c]
            df_merged = pd.concat([df_merged, other_df[other_feats]], axis=1)

        # renombrar features finales a feat_0 ... feat_K-1
        all_feat_cols = [c for c in df_merged.columns if "__" in c]
        new_names = [f"feat_{i}" for i in range(len(all_feat_cols))]
        rename_map = dict(zip(all_feat_cols, new_names))
        df_merged = df_merged.rename(columns=rename_map)

        # ordenar columnas: IDs + feats
        df_out = pd.concat([df_merged[ID_COLS], df_merged[new_names]], axis=1)

        out_path = os.path.join(output_dir, f"{nombre_grupo}_{mode}.csv")
        df_out.to_csv(out_path, index=False)
        print(f" Guardado: {out_path}  ({df_out.shape[0]} filas, {len(new_names)} features)")

    elif mode == "mean":
        # todas las fuentes deben tener la MISMA dimensión de features
        if len(set(feats_counts)) != 1:
            raise ValueError(f"No se puede promediar: dimensiones distintas {feats_counts}")

        stacked = np.stack(mats, axis=0)   # [M, N, D]
        mean_feats = stacked.mean(axis=0)  # [N, D]

        feat_names = [f"feat_{i}" for i in range(mean_feats.shape[1])]
        out = pd.concat(
            [ids_ref.reset_index(drop=True),
             pd.DataFrame(mean_feats, columns=feat_names)],
            axis=1
        )

        out_path = os.path.join(output_dir, f"{nombre_grupo}_{mode}.csv")
        out.to_csv(out_path, index=False)
        print(f" Guardado: {out_path}  ({out.shape[0]} filas, {mean_feats.shape[1]} features)")

# Ejecuta ambos modos si quieres:
for nombre_grupo, lista_csv in compatible_groups.items():
    combinar_grupo(nombre_grupo, lista_csv, mode="concat")
    try:
        combinar_grupo(nombre_grupo, lista_csv, mode="mean")
    except ValueError as e:
        print(f" Saltando mean para '{nombre_grupo}': {e}")


### V4: Alineación por steps mínimos y combinación entre extractores (OUMVLP, GREW)

Esta celda permite combinar embeddings procedentes de extractores distintos (p. ej., Baseline, GaitPart, GaitSet, GaitGL) dentro de un mismo grupo (OUMVLP, GREW, etc.), incluso cuando cada extractor produce un número diferente de steps. Para hacerlo, primero se identifica target_steps como el mínimo número de steps del grupo y, a continuación, cada embedding se reduce a esa resolución mediante un mapeo proporcional en el intervalo [0,1]: cada step destino agrega por media los steps originales que caen en su franja (sin interpolación continua). Este proceso mantiene la coherencia anatómica relativa (de arriba a abajo) y genera secuencias con longitud común.

Una vez reducidos, los embeddings se sincronizan tomando la intersección de claves (video_ID,step) para asegurar que todas las fuentes están perfectamente alineadas. Por último, se realiza la fusión final paso a paso con dos modos:

**concat**: concatenación horizontal de todas las features reducidas (máxima información, mayor dimensionalidad).

**mean**: promedio entre embeddings tras recortar a la dimensión mínima común (trim a min_d) cuando los tamaños no coinciden.

El resultado se exporta como un único CSV combinado por grupo en Gait_Embeddings_Reduced/ (p. ej., OUMVLP_mean.csv, GREW_mean.csv), con columnas feat_0…feat_{K-1} y manteniendo la estructura secuencial necesaria para LSTM/TCN/Transformer.

In [ ]:
import os
import numpy as np
import pandas as pd
from functools import reduce

INPUT_DIR  = "Gait_Embeddings_good/"
OUTPUT_DIR = "Gait_Embeddings_Reduced/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

ID_COLS = ["video_ID", "pitch_side", "kicker_foot", "shoot_zone"]

def franjas_proporcionales_indices(s_orig: int, target_steps: int):
    """Mapea cada franja destino g a una lista de índices de steps originales (sin interpolar)."""
    idxs = []
    orig_edges   = np.linspace(0, 1, s_orig + 1)
    target_edges = np.linspace(0, 1, target_steps + 1)
    target_centers = 0.5 * (target_edges[:-1] + target_edges[1:])
    orig_centers   = (np.arange(s_orig) + 0.5) / s_orig
    for g in range(target_steps):
        a, b = target_edges[g], target_edges[g + 1]
        mask_full = (orig_edges[:-1] >= a) & (orig_edges[1:] <= b)
        sel = np.where(mask_full)[0].tolist()
        if not sel:
            nearest = int(np.argmin(np.abs(orig_centers - target_centers[g])))
            sel = [nearest]
        idxs.append(sel)
    return idxs


def reduce_to_target_steps(df: pd.DataFrame, target_steps: int) -> pd.DataFrame:
    feat_cols = [c for c in df.columns if c.startswith("feat_")]
    s_orig = df["step"].nunique()
    map_franjas = franjas_proporcionales_indices(s_orig, target_steps)

    out_rows = []
    for vid, g in df.groupby("video_ID"):
        meta = g[["video_ID","pitch_side","kicker_foot","shoot_zone"]].iloc[0].to_dict()
        g = g.sort_values("step")
        for g_idx, orig_idx_list in enumerate(map_franjas):
            block = g[g["step"].isin(orig_idx_list)]
            feats_vec = block[feat_cols].mean().to_numpy(dtype=np.float32)  # <- solo mean
            row = {"step": g_idx, **meta}
            for j, f in enumerate(feat_cols):
                row[f] = feats_vec[j]
            out_rows.append(row)

    return pd.DataFrame(out_rows).sort_values(["video_ID","step"]).reset_index(drop=True)


def combine_dataset(dataset_name: str, embeddings: list, agg: str = "mean", combine_mode: str = "concat") -> str:
    """
    Pipeline:
      1) Detecta target_steps = mínimo del grupo.
      2) Reduce cada embedding a target_steps (en memoria).
      3) Sincroniza por intersección de (video_ID, step).
      4) Combina paso a paso: 'concat' (apilar) o 'mean' (promedio con trim a minD).
      5) Guarda SOLO el combinado final.
    """
    # 1) target_steps
    steps_counts = {}
    for emb in embeddings:
        df_head = pd.read_csv(os.path.join(INPUT_DIR, f"{emb}.csv"), nrows=2000)
        steps_counts[emb] = df_head["step"].nunique()
    target_steps = min(steps_counts.values())
    print(f"\n🔹 {dataset_name}: target_steps = {target_steps}  (min {steps_counts})")

    # 2) reducir en memoria
    reduced_list = []
    for emb in embeddings:
        df = pd.read_csv(os.path.join(INPUT_DIR, f"{emb}.csv")).sort_values(["video_ID","step"]).reset_index(drop=True)
        red = reduce_to_target_steps(df, target_steps=target_steps)
        # renombra feats para saber su bloque si luego concatenamos
        feat_cols = [c for c in red.columns if c.startswith("feat_")]
        red = red.rename(columns={c: f"{emb}__{i}" for i, c in enumerate(feat_cols)})
        reduced_list.append(red)
        print(f"   Reducido en memoria: {emb}  ({steps_counts[emb]}→{target_steps})")

    # 3) sincronizar por intersección (video_ID, step)
    keys = ["video_ID", "step"]
    common = reduce(lambda a,b: pd.merge(a[keys].drop_duplicates(), b[keys].drop_duplicates(), on=keys, how="inner"),
                    reduced_list)
    if common.empty:
        raise RuntimeError("No hay intersección de (video_ID, step) entre los modelos.")
    synced = []
    for red in reduced_list:
        synced.append(pd.merge(common, red, on=keys, how="inner").sort_values(keys).reset_index(drop=True))

    # 4) combinar
    ids_ref = synced[0][keys + ["pitch_side","kicker_foot","shoot_zone"]].copy()
    blocks = []
    dims = []
    for dfb in synced:
        feat_cols = [c for c in dfb.columns if "__" in c]
        X = dfb[feat_cols].to_numpy(dtype=np.float32)
        blocks.append(X); dims.append(X.shape[1])

    if combine_mode == "concat":
        fused = np.concatenate(blocks, axis=1).astype(np.float32)
    elif combine_mode == "mean":
        min_d = min(dims)
        blocks_trim = [b[:, :min_d] for b in blocks]
        fused = np.mean(np.stack(blocks_trim, axis=0), axis=0).astype(np.float32)
    else:
        raise ValueError("combine_mode debe ser 'concat' o 'mean'")

    feat_names = [f"feat_{i}" for i in range(fused.shape[1])]
    out_df = pd.concat([ids_ref.reset_index(drop=True), pd.DataFrame(fused, columns=feat_names)], axis=1)

    # out_path = os.path.join(OUTPUT_DIR, f"{dataset_name}_COMBINED_{target_steps}steps_{agg}_{combine_mode}.csv")
    out_path = os.path.join(OUTPUT_DIR, f"{dataset_name}_{combine_mode}.csv")

    out_df.to_csv(out_path, index=False)
    print(f" Combinado guardado: {out_path}  (videos={out_df['video_ID'].nunique()}, steps={target_steps}, dims={fused.shape[1]})")
    return out_path


# === Ejemplos ===
combine_dataset("OUMVLP",
    ["baseline_OUMVLP","gaitpart_OUMVLP","gaitset_OUMVLP","gaitgl_OUMVLP"], combine_mode="mean")

combine_dataset("GREW",
    ["gaitpart_GREW","gaitset_GREW","gaitgl_GREW"], combine_mode="mean")


### V5: Embeddings recortados (Top-K% de steps).

A partir del ranking consensuado obtenido en la etapa de selección, esta celda construye versiones recortadas de cada embedding conservando únicamente el Top-K% de steps más informativos según hybrid_global_score. Para cada embedding se calcula top_k = ceil(ratio · n_steps) (garantizando al menos 1 step), se seleccionan los pasos con mayor puntuación y se filtra el CSV original manteniendo el orden video_ID–step. Los nuevos embeddings se guardan en Gait_Embeddings_TopSteps/ con sufijo _TOP{porcentaje} (por ejemplo, _TOP50), y se utilizan posteriormente como variantes V5 para comparar rendimiento frente al embedding completo.

In [ ]:
import os
import math
import pandas as pd

# Directorios
INPUT_DIR = "Gait_Embeddings_good/"
STEPSEL_DIR = "Step_Analysis_Selection/"
OUTPUT_DIR = "Gait_Embeddings_TopSteps/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def load_top_steps(
    embedding_name: str,
    ratio: float,
    ranking_col: str = "hybrid_global_score"
):
    """
    Selecciona el top-X% de steps según el ranking híbrido.
    Devuelve los steps ordenados (ascendente).
    """
    scores_csv = os.path.join(
        STEPSEL_DIR,
        embedding_name,
        "step_selection_hybrid_final.csv"
    )

    df_scores = pd.read_csv(scores_csv)

    n_steps = len(df_scores)
    top_k = max(1, math.ceil(n_steps * ratio))

    df_top = (
        df_scores
        .sort_values(ranking_col, ascending=False)
        .head(top_k)
    )

    return sorted(df_top["step"].tolist()), top_k, n_steps


def build_topsteps_embedding(
    embedding_name: str,
    ratio: float
):
    """
    Construye un embedding recortado usando el top-X% de steps.
    """
    in_csv = os.path.join(INPUT_DIR, f"{embedding_name}.csv")
    df = pd.read_csv(in_csv)

    top_steps, top_k, total_steps = load_top_steps(
        embedding_name, ratio
    )

    df_out = (
        df[df["step"].isin(top_steps)]
        .sort_values(["video_ID", "step"])
        .reset_index(drop=True)
    )

    ratio_pct = int(ratio * 100)
    out_path = os.path.join(
        OUTPUT_DIR,
        f"{embedding_name}_TOP{ratio_pct}.csv"
    )

    df_out.to_csv(out_path, index=False)

    print(
        f" {embedding_name}: "
        f"{top_k}/{total_steps} steps "
        f"({ratio_pct}%) → {out_path}"
    )


# === Ejecución ===
RATIOS = [0.5]

for csv_file in os.listdir(INPUT_DIR):
    if not csv_file.endswith(".csv"):
        continue

    embedding_name = csv_file.replace(".csv", "")

    for r in RATIOS:
        build_topsteps_embedding(embedding_name, r)


### V5: Variante combinación de los peores steps (Low% steps)

In [ ]:
import os
import math
import pandas as pd

# Directorios
INPUT_DIR = "Gait_Embeddings_good/"
STEPSEL_DIR = "Step_Analysis_Selection/"
# He cambiado el nombre del directorio de salida para no mezclar con los "Top"
OUTPUT_DIR = "Gait_Embeddings_WorstSteps/" 
os.makedirs(OUTPUT_DIR, exist_ok=True)

def load_worst_steps(
    embedding_name: str,
    ratio: float,
    ranking_col: str = "hybrid_global_score"
):
    """
    Selecciona el bottom-X% de steps (los peores) según el ranking híbrido.
    Devuelve los steps ordenados (ascendente por número de step para el filtrado posterior).
    """
    scores_csv = os.path.join(
        STEPSEL_DIR,
        embedding_name,
        "step_selection_hybrid_final.csv"
    )

    df_scores = pd.read_csv(scores_csv)

    n_steps = len(df_scores)
    top_k = max(1, math.ceil(n_steps * ratio))

    # CAMBIO PRINCIPAL AQUÍ:
    # ascending=True ordena de menor a mayor (peores puntuaciones primero)
    df_worst = (
        df_scores
        .sort_values(ranking_col, ascending=True) 
        .head(top_k)
    )

    return sorted(df_worst["step"].tolist()), top_k, n_steps


def build_worststeps_embedding(
    embedding_name: str,
    ratio: float
):
    """
    Construye un embedding recortado usando el bottom-X% de steps (peores).
    """
    in_csv = os.path.join(INPUT_DIR, f"{embedding_name}.csv")
    
    # Verificación de seguridad por si el archivo no existe
    if not os.path.exists(in_csv):
        print(f" Archivo no encontrado: {in_csv}")
        return

    df = pd.read_csv(in_csv)

    # Llamamos a la función de "worst" steps
    worst_steps, k_selected, total_steps = load_worst_steps(
        embedding_name, ratio
    )

    df_out = (
        df[df["step"].isin(worst_steps)]
        .sort_values(["video_ID", "step"])
        .reset_index(drop=True)
    )

    ratio_pct = int(ratio * 100)
    
    # Cambio en el nombre del archivo de salida para identificarlo como WORST
    out_path = os.path.join(
        OUTPUT_DIR,
        f"{embedding_name}_LOW{ratio_pct}.csv"
    )

    df_out.to_csv(out_path, index=False)

    print(
        f" {embedding_name}: "
        f"{k_selected}/{total_steps} steps (Peores) "
        f"({ratio_pct}%) → {out_path}"
    )


# === Ejecución ===
RATIOS = [0.3, 0.7]

for csv_file in os.listdir(INPUT_DIR):
    if not csv_file.endswith(".csv"):
        continue

    embedding_name = csv_file.replace(".csv", "")

    for r in RATIOS:
        build_worststeps_embedding(embedding_name, r)
